In [ ]:
## 🚀 SCRIPT TỰ ĐỘNG HÓA TRAINING T5 TRÊN KAGGLE
## Trước khi chạy: Upload dataset chứa các file .py lên Kaggle

import os
import shutil
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==================================================================================
# CẤU HÌNH THỰC NGHIỆM (BẠN CHỈ CẦN SỬA Ở ĐÂY)
# ==================================================================================
CONFIG = {
    "model_name": "t5-small",              # hoặc "t5-base"
    "epochs": 100,                         # Số epoch
    "batch_size": 128,                     # Batch size
    "train_samples": 1000,                 # Số mẫu train (mặc định trong generate_data)
    "format": "10e-based",                 # Format: decimal, 10e-based, character, etc.
    "digits": [2, 5, 10, 15, 20, 25, 30],  # Các độ dài số để test
    "num_runs": 5,                         # Số lần chạy với seed khác nhau (1-5)
    "output_dir": "outputs/run_full_auto", # Thư mục lưu kết quả
    "seed": 42,                            # Seed bắt đầu
    "only_addition": True,                 # True = chỉ phép cộng, False = cộng + trừ
    "max_seq_length": 512                  # Độ dài tối đa của sequence
}
# ==================================================================================

def run_command(command):
    """Chạy lệnh shell và kiểm tra kết quả"""
    print(f"⚡ Running: {command[:80]}...")
    exit_code = os.system(command)
    if exit_code != 0:
        print(f"❌ ERROR: Command failed with exit code {exit_code}")
        print(f"   Command: {command}")
        return False
    return True

print("=" * 80)
print("🚀 BẮT ĐẦU QUÁ TRÌNH TỰ ĐỘNG HÓA TRAINING T5")
print("=" * 80)

# ----------------------------------------------------------------------------------
# BƯỚC 1: TỰ ĐỘNG TÌM VÀ COPY CODE TỪ /kaggle/input
# ----------------------------------------------------------------------------------
print("\n[1/6] 📁 Copying Python files from input...")
source_root = "/kaggle/input"
dest_dir = "/kaggle/working"
py_files_count = 0

for root, dirs, files in os.walk(source_root):
    for file in files:
        if file.endswith(".py"):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dest_dir, file)
            shutil.copy(src_path, dst_path)
            print(f"   ✓ Copied: {file}")
            py_files_count += 1

if py_files_count == 0:
    raise FileNotFoundError("❌ CRITICAL: Không tìm thấy file .py nào! Hãy upload dataset code trước.")

print(f"✅ Copied {py_files_count} Python files successfully\n")

# ----------------------------------------------------------------------------------
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN
# ----------------------------------------------------------------------------------
print("[2/6] 📦 Installing required packages...")
if not run_command("pip install -q pytorch-lightning transformers num2words protobuf==3.20.3 libraft-cu12>=25.6.0 rmm-cu12>=25.6.0"):
    raise RuntimeError("❌ Failed to install packages!")
print("✅ Packages installed successfully\n")

# ----------------------------------------------------------------------------------
# BƯỚC 3: KIỂM TRA CÁC FILE CẦN THIẾT
# ----------------------------------------------------------------------------------
print("[3/6] 🔍 Checking required files...")
required_files = ["generate_data.py", "utils.py", "train.py", "dataset.py", "config.py"]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(f"❌ Missing files: {', '.join(missing_files)}")

print("✅ All required files found\n")

# ----------------------------------------------------------------------------------
# BƯỚC 4: TẠO THƯ MỤC VÀ SINH DỮ LIỆU
# ----------------------------------------------------------------------------------
print("[4/6] 📊 Generating datasets...")
run_command("mkdir -p data outputs")

# Tạo dữ liệu với experiment mode (tự động tạo cho tất cả digits)
gen_flags = "--only_addition" if CONFIG["only_addition"] else ""
gen_cmd = (
    f"python generate_data.py "
    f"--mode experiment "
    f"--base_path data/experiment "
    f"--seed {CONFIG['seed']} "
    f"--format {CONFIG['format']} "
    f"--digit_steps {' '.join(map(str, CONFIG['digits']))} "
    f"{gen_flags}"
)

if not run_command(gen_cmd):
    raise RuntimeError("❌ Data generation failed!")

print("✅ Data generated successfully\n")

# ----------------------------------------------------------------------------------
# BƯỚC 5: HUẤN LUYỆN MODEL CHO TẤT CẢ CÁC CẤU HÌNH
# ----------------------------------------------------------------------------------
print(f"[5/6] 🏋️ Training model '{CONFIG['model_name']}'...")
print(f"   Digits to test: {CONFIG['digits']}")
print(f"   Runs per digit: {CONFIG['num_runs']}")
print(f"   Total trainings: {len(CONFIG['digits']) * CONFIG['num_runs']}\n")

training_count = 0
success_count = 0
failed_runs = []

for d in CONFIG['digits']:
    print(f"\n{'='*80}")
    print(f"🔢 DIGITS: {d}")
    print(f"{'='*80}")
    
    for run_id in range(1, CONFIG['num_runs'] + 1):  # 1 đến 5 (hoặc num_runs)
        training_count += 1
        print(f"\n[Run {run_id}/{CONFIG['num_runs']}] Training with {d} digits...")
        
        # Tên file data theo generate_data.py (train_1.json, val_1.json, test_1.json)
        train_file = f"data/experiment/digits_{d}/train_{run_id}.json"
        val_file = f"data/experiment/digits_{d}/val_{run_id}.json"
        test_file = f"data/experiment/digits_{d}/test_{run_id}.json"
        
        # Kiểm tra file tồn tại
        if not all(os.path.exists(f) for f in [train_file, val_file, test_file]):
            print(f"   ⚠️  Skipping: Data files not found for digits={d}, run={run_id}")
            failed_runs.append(f"digits_{d}_run_{run_id} (missing data)")
            continue
        
        output_path = f"{CONFIG['output_dir']}/digits_{d}/run_{run_id}"
        
        train_cmd = (
            f"python train.py "
            f"--train_file {train_file} "
            f"--val_file {val_file} "
            f"--test_file {test_file} "
            f"--output_dir {output_path} "
            f"--model_name_or_path {CONFIG['model_name']} "
            f"--format {CONFIG['format']} "
            f"--train_size_log {CONFIG['train_samples']} "
            f"--sampling_strategy balanced "
            f"--epochs {CONFIG['epochs']} "
            f"--train_batch_size {CONFIG['batch_size']} "
            f"--val_batch_size {CONFIG['batch_size']} "
            f"--lr 3e-4 "
            f"--seed {CONFIG['seed'] + run_id} "
            f"--accelerator gpu "
            f"--devices 1 "
            f"--max_seq_length {CONFIG['max_seq_length']}"
        )
        
        if run_command(train_cmd):
            print(f"   ✅ Training completed successfully!")
            success_count += 1
        else:
            print(f"   ❌ Training failed!")
            failed_runs.append(f"digits_{d}_run_{run_id}")

print(f"\n{'='*80}")
print(f"📊 TRAINING SUMMARY")
print(f"{'='*80}")
print(f"Total trainings: {training_count}")
print(f"Successful: {success_count}")
print(f"Failed: {training_count - success_count}")
if failed_runs:
    print(f"\nFailed runs:")
    for run in failed_runs:
        print(f"   - {run}")
print(f"{'='*80}\n")

# ----------------------------------------------------------------------------------
# BƯỚC 6: NÉN KẾT QUẢ
# ----------------------------------------------------------------------------------
print("[6/6] 📦 Compressing results...")

# Kiểm tra thư mục output có tồn tại không
if not os.path.exists(CONFIG['output_dir']):
    print("⚠️  Warning: Output directory not found, creating empty archive")
    run_command(f"mkdir -p {CONFIG['output_dir']}")

zip_cmd = f"zip -r ket_qua_full.zip {CONFIG['output_dir']}"
if run_command(zip_cmd):
    print("✅ Results compressed successfully!")
    
    # Kiểm tra kích thước file
    if os.path.exists("ket_qua_full.zip"):
        size_mb = os.path.getsize("ket_qua_full.zip") / (1024 * 1024)
        print(f"   📦 Archive size: {size_mb:.2f} MB")
else:
    print("❌ Compression failed!")

print("\n" + "="*80)
print("🎉 HOÀN THÀNH TẤT CẢ! HÃY TẢI FILE 'ket_qua_full.zip' VỀ")
print("="*80)
print(f"\n💡 Tips:")
print(f"   - Kết quả được lưu trong thư mục: {CONFIG['output_dir']}")
print(f"   - Mỗi run có file results.json và checkpoint models")
print(f"   - Xem log để kiểm tra accuracy của từng experiment")

🚀 BẮT ĐẦU QUÁ TRÌNH TỰ ĐỘNG HÓA TRAINING T5

[1/6] 📁 Copying Python files from input...


FileNotFoundError: ❌ CRITICAL: Không tìm thấy file .py nào! Hãy upload dataset code trước.